# Generic CSV Data Understanding Notebook

This notebook is **data understanding only**. It does not perform preprocessing or ML training.

It supports:

- Multiple CSV files in one run
- Independent, detailed analysis of each file
- Shape, schema, dtypes, missingness, uniqueness and duplicates
- Numeric and categorical profiling
- Temporal-column detection
- Outlier screening
- Correlation screening
- Likely ID/key detection
- ML-readiness warnings
- A separate two-file comparison section for future merging

The output should be used to decide the later preprocessing, EDA, feature engineering and ML strategy.

In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 100)

print("Environment ready.")

Environment ready.


## 1. Select CSV files

You can provide multiple files. They will **not** automatically be merged.

Example:

```text
data/crime_women.csv, data/crime_children.csv, data/crime_cyber.csv
```

In [2]:
# Option A: define paths directly.
CSV_FILES = [
    r"C:\\Users\\nishantwhat\\CODE\\AUCML_DATA\\districtwise-cyber-crimes-2017-onwards.csv",
    r"C:\\Users\\nishantwhat\\CODE\\AUCML_DATA\\districtwise-cyber-crimes-2024-onwards.csv",
]

# Option B: enter paths interactively if CSV_FILES is empty.
if not CSV_FILES:
    raw = input("Enter CSV file paths separated by commas: ").strip()
    CSV_FILES = [
        x.strip().strip('"').strip("'")
        for x in raw.split(",")
        if x.strip()
    ]

CSV_FILES = [str(Path(p)) for p in CSV_FILES]

print(f"Selected {len(CSV_FILES)} file(s):")
for path in CSV_FILES:
    print(f"  {path} | exists={Path(path).exists()}")

Selected 2 file(s):
  C:\Users\nishantwhat\CODE\AUCML_DATA\districtwise-cyber-crimes-2017-onwards.csv | exists=True
  C:\Users\nishantwhat\CODE\AUCML_DATA\districtwise-cyber-crimes-2024-onwards.csv | exists=True


In [3]:
def load_csv(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix.lower() != ".csv":
        raise ValueError(f"Expected CSV file, got: {path}")

    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as first_error:
        try:
            return pd.read_csv(path, encoding="latin-1", low_memory=False)
        except Exception as second_error:
            raise RuntimeError(
                f"Could not read {path}. "
                f"Normal read failed: {first_error}. "
                f"Fallback read failed: {second_error}."
            )

datasets = {}

for path in CSV_FILES:
    try:
        name = Path(path).stem
        datasets[name] = load_csv(path)
        print(f"Loaded: {name} -> {datasets[name].shape}")
    except Exception as exc:
        print(f"ERROR: {path}\n  {exc}")

Loaded: districtwise-cyber-crimes-2017-onwards -> (5322, 48)
Loaded: districtwise-cyber-crimes-2024-onwards -> (964, 56)


## 2. Column classification helpers

In [4]:
def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def likely_id_column(series):
    name = normalize_name(series.name)
    n = len(series)

    if n == 0:
        return False

    unique_ratio = series.nunique(dropna=True) / n

    name_hint = bool(
        re.search(r"(^id$|_id$|^id_|code$|_code$|key$|_key$)", name)
    )

    high_unique_text = (
        unique_ratio >= 0.95
        and (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        )
    )

    return name_hint or high_unique_text


def likely_datetime_column(series):
    name = normalize_name(series.name)
    name_hint = bool(
        re.search(r"date|time|year|month|day|timestamp", name)
    )

    if pd.api.types.is_datetime64_any_dtype(series):
        return True

    if pd.api.types.is_numeric_dtype(series) and not name_hint:
        return False

    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        sample = series.dropna().astype(str).head(300)

        if len(sample) == 0:
            return name_hint

        parsed = pd.to_datetime(sample, errors="coerce")
        parse_ratio = parsed.notna().mean()

        return name_hint or parse_ratio >= 0.85

    return name_hint


def column_role(series):
    if likely_datetime_column(series):
        return "datetime / temporal"

    if likely_id_column(series):
        return "likely ID / key"

    if pd.api.types.is_numeric_dtype(series):
        return "numeric"

    if pd.api.types.is_bool_dtype(series):
        return "boolean"

    return "categorical / text"

## 3. Single-file analysis

In [5]:
def analyze_single_file(df, file_name, top_n=10):
    print("=" * 110)
    print(f"DATASET: {file_name}")
    print("=" * 110)

    # ------------------------------------------------------------
    # 1. Overview
    # ------------------------------------------------------------
    print("\n[1] OVERVIEW")

    duplicate_count = int(df.duplicated().sum())
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2

    print(f"Rows:             {len(df):,}")
    print(f"Columns:          {df.shape[1]:,}")
    print(f"Memory:           {memory_mb:.2f} MB")
    print(f"Duplicate rows:   {duplicate_count:,} ({duplicate_count / len(df) * 100:.2f}%)"
          if len(df) else "Duplicate rows: 0")

    # ------------------------------------------------------------
    # 2. Sample
    # ------------------------------------------------------------
    print("\n[2] SAMPLE")
    display(df.head(top_n))

    # ------------------------------------------------------------
    # 3. Schema
    # ------------------------------------------------------------
    print("\n[3] SCHEMA / COLUMN PROFILE")

    rows = []

    for column in df.columns:
        series = df[column]
        non_null = int(series.notna().sum())
        unique = int(series.nunique(dropna=True))

        rows.append({
            "column": column,
            "dtype": str(series.dtype),
            "role": column_role(series),
            "rows": len(series),
            "non_null": non_null,
            "missing": int(series.isna().sum()),
            "missing_%": round(series.isna().mean() * 100, 2),
            "unique": unique,
            "unique_%": round(unique / len(series) * 100, 2) if len(series) else 0,
            "duplicated_values": non_null - unique,
        })

    profile = pd.DataFrame(rows).sort_values(
        ["missing_%", "column"],
        ascending=[False, True]
    )

    display(profile)

    # ------------------------------------------------------------
    # 4. Numeric analysis
    # ------------------------------------------------------------
    numeric = df.select_dtypes(include=np.number)

    print("\n[4] NUMERIC ANALYSIS")

    if numeric.shape[1] > 0:
        display(
            numeric.describe(
                percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
            ).T
        )
    else:
        print("No numeric columns detected.")

    # ------------------------------------------------------------
    # 5. Categorical analysis
    # ------------------------------------------------------------
    categorical = df.select_dtypes(
        include=["object", "string", "category", "bool"]
    )

    print("\n[5] CATEGORICAL / TEXT ANALYSIS")

    if categorical.shape[1] > 0:
        cat_rows = []

        for column in categorical.columns:
            series = categorical[column]
            value_counts = series.value_counts(dropna=False)

            top_value = value_counts.index[0] if len(value_counts) else None
            top_count = int(value_counts.iloc[0]) if len(value_counts) else 0

            cat_rows.append({
                "column": column,
                "unique": int(series.nunique(dropna=True)),
                "missing_%": round(series.isna().mean() * 100, 2),
                "top_value": str(top_value)[:100],
                "top_frequency": top_count,
                "top_frequency_%": round(
                    top_count / len(series) * 100, 2
                ) if len(series) else 0,
            })

        display(
            pd.DataFrame(cat_rows).sort_values(
                "unique", ascending=False
            )
        )
    else:
        print("No categorical/text columns detected.")

    # ------------------------------------------------------------
    # 6. Constant / near-constant columns
    # ------------------------------------------------------------
    print("\n[6] LOW-VARIANCE / CONSTANT COLUMNS")

    low_variance = []

    for column in df.columns:
        series = df[column]
        frequencies = series.value_counts(
            dropna=False,
            normalize=True
        )

        top_frequency = frequencies.iloc[0] if len(frequencies) else 0

        if series.nunique(dropna=False) <= 1 or top_frequency >= 0.995:
            low_variance.append({
                "column": column,
                "distinct_including_null": int(series.nunique(dropna=False)),
                "top_value_%": round(top_frequency * 100, 2),
            })

    if low_variance:
        display(pd.DataFrame(low_variance))
    else:
        print("No obvious constant / near-constant columns.")

    # ------------------------------------------------------------
    # 7. Outlier screening
    # ------------------------------------------------------------
    print("\n[7] OUTLIER SCREENING — IQR RULE")

    outlier_rows = []

    for column in numeric.columns:
        series = numeric[column].dropna()

        if len(series) < 4:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            count = 0
        else:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = int(((series < lower) | (series > upper)).sum())

        outlier_rows.append({
            "column": column,
            "outliers": count,
            "outlier_%": round(count / len(series) * 100, 2),
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
        })

    if outlier_rows:
        display(
            pd.DataFrame(outlier_rows).sort_values(
                "outlier_%", ascending=False
            )
        )
    else:
        print("No suitable numeric columns for IQR screening.")

    # ------------------------------------------------------------
    # 8. Correlations
    # ------------------------------------------------------------
    print("\n[8] NUMERIC CORRELATION SCREEN")

    if numeric.shape[1] >= 2:
        correlation = numeric.corr(numeric_only=True)

        pairs = []
        columns = list(correlation.columns)

        for i in range(len(columns)):
            for j in range(i + 1, len(columns)):
                value = correlation.iloc[i, j]

                if pd.notna(value):
                    pairs.append({
                        "feature_1": columns[i],
                        "feature_2": columns[j],
                        "correlation": value,
                        "abs_correlation": abs(value),
                    })

        correlation_pairs = pd.DataFrame(pairs)

        if not correlation_pairs.empty:
            display(
                correlation_pairs.sort_values(
                    "abs_correlation",
                    ascending=False
                ).head(20)
            )
        else:
            print("No valid correlation pairs.")
    else:
        print("Need at least two numeric columns.")

    # ------------------------------------------------------------
    # 9. Temporal columns
    # ------------------------------------------------------------
    print("\n[9] TEMPORAL COLUMN SCREEN")

    temporal_columns = [
        column
        for column in df.columns
        if likely_datetime_column(df[column])
    ]

    if temporal_columns:
        print("Potential temporal columns:", temporal_columns)

        for column in temporal_columns:
            parsed = pd.to_datetime(df[column], errors="coerce")
            valid = parsed.dropna()

            if len(valid):
                print(
                    f"  {column}: {valid.min()} -> {valid.max()} | "
                    f"parse success={len(valid) / len(df) * 100:.1f}%"
                )
    else:
        print("No obvious temporal columns detected.")

    # ------------------------------------------------------------
    # 10. ML-readiness observations
    # ------------------------------------------------------------
    print("\n[10] ML-READINESS OBSERVATIONS — NOT ACTIONS")

    observations = []

    for column in df.columns:
        series = df[column]
        missing_ratio = series.isna().mean()
        unique = series.nunique(dropna=True)

        if missing_ratio > 0:
            observations.append(
                f"{column}: {missing_ratio * 100:.1f}% missing values"
            )

        if unique == 1:
            observations.append(
                f"{column}: constant column"
            )

        if len(df) and unique / len(df) > 0.98:
            observations.append(
                f"{column}: very high cardinality / possible identifier"
            )

    if df.duplicated().any():
        observations.append(
            f"{int(df.duplicated().sum()):,} duplicate rows detected"
        )

    if observations:
        for observation in observations[:50]:
            print(" -", observation)
    else:
        print("No immediate structural warnings detected.")

    return profile


profiles = {}

for name, df in datasets.items():
    profiles[name] = analyze_single_file(df, name)

DATASET: districtwise-cyber-crimes-2017-onwards

[1] OVERVIEW
Rows:             5,322
Columns:          48
Memory:           2.72 MB
Duplicate rows:   0 (0.00%)

[2] SAMPLE


,id,year,state_name,state_code,district_name,district_code,registration_circles,tampering_computer_source_documents,ransom_ware,offences_other_than_ransom_ware,dishonestly_recv_stolen_cmp_resrc_or_comm_device,identity_theft,cheating_by_personation_by_using_computer_resource,violation_of_privacy,cyber_terrorism,pub_or_trans_obscene_material_in_electronic_form,pub_or_trans_mtrl_cont_sxly_explct_act_elect_frm,pub_or_trans_mtrl_dpct_chl_sxly_explct_elect_frm,presrv_and_retention_of_info_by_intermediaries,other_sections_it_act,interception_or_monitoring_or_decryption_of_info,un_athryz_access_atmpt_access_prct_comp_sys,abetment_to_commit_offences,attempt_to_commit_offences,other_sections_of_it_act,abetment_of_suicide_online,cyber_stalking_bullying_of_women_children,data_theft,credit_card_debit_card_fraud,atms_fraud,online_banking_fraud,otp_frauds,other_frauds,cheating,forgery,defamation_morphing,fake_profile,currency_counterfeiting,stamps_counterfeiting,cyber_blackmailing_threatening,fake_news_on_social_media,other_offences,total_offences_under_ip,gambling_act,lotteries_act,copy_right_act,trade_marks_act,other_sll_crimes
0,0,2017,Jammu And Kashmir,1,Anantnag,1,Anantnag,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,2017,Jammu And Kashmir,1,Pulwama,11,Awantipora,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,2017,Jammu And Kashmir,1,Bandipora,623,Bandipora,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,2017,Jammu And Kashmir,1,Baramulla,3,Baramulla,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,2017,Jammu And Kashmir,1,Budgam,2,Budgam,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,5,2017,Jammu And Kashmir,1,Jammu,5,Crime Jammu,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,3,0,0,0,0,0
6,6,2017,Jammu And Kashmir,1,Srinagar,13,Crime Srinagar,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,7,2017,Jammu And Kashmir,1,Doda,4,Doda,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,8,2017,Jammu And Kashmir,1,Ganderbal,626,Ganderbal,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,9,2017,Jammu And Kashmir,1,Kupwara,8,Handwara,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



[3] SCHEMA / COLUMN PROFILE


,column,dtype,role,rows,non_null,missing,missing_%,unique,unique_%,duplicated_values
25,abetment_of_suicide_online,int64,numeric,5322,5322,0,0.0,4,0.08,5318
22,abetment_to_commit_offences,int64,numeric,5322,5322,0,0.0,4,0.08,5318
29,atms_fraud,int64,numeric,5322,5322,0,0.0,74,1.39,5248
23,attempt_to_commit_offences,int64,numeric,5322,5322,0,0.0,6,0.11,5316
33,cheating,int64,numeric,5322,5322,0,0.0,106,1.99,5216
12,cheating_by_personation_by_using_computer_resource,int64,numeric,5322,5322,0,0.0,134,2.52,5188
45,copy_right_act,int64,numeric,5322,5322,0,0.0,17,0.32,5305
28,credit_card_debit_card_fraud,int64,numeric,5322,5322,0,0.0,52,0.98,5270
37,currency_counterfeiting,int64,numeric,5322,5322,0,0.0,4,0.08,5318
39,cyber_blackmailing_threatening,int64,numeric,5322,5322,0,0.0,30,0.56,5292



[4] NUMERIC ANALYSIS


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
id,5322.0,2660.500000,1536.473397,0.0,53.21,266.05,1330.25,2660.5,3990.75,5054.95,5267.79,5321.0
year,5322.0,2019.555054,1.709628,2017.0,2017.00,2017.00,2018.00,2020.0,2021.00,2022.00,2022.00,2022.0
state_code,5322.0,17.719466,9.889335,1.0,1.00,2.00,9.00,19.0,24.00,33.00,36.00,38.0
district_code,5322.0,367.008643,386.280327,1.0,5.00,30.00,171.00,356.0,536.75,688.00,734.00,9999.0
tampering_computer_source_documents,5322.0,0.210635,3.557165,0.0,0.00,0.00,0.00,0.0,0.00,0.00,3.00,187.0
ransom_ware,5322.0,0.915445,15.983453,0.0,0.00,0.00,0.00,0.0,0.00,0.00,11.00,578.0
offences_other_than_ransom_ware,5322.0,3.616873,31.477140,0.0,0.00,0.00,0.00,0.0,0.00,10.00,74.79,1087.0
dishonestly_recv_stolen_cmp_resrc_or_comm_device,5322.0,0.422585,3.589346,0.0,0.00,0.00,0.00,0.0,0.00,1.00,8.00,99.0
identity_theft,5322.0,7.069335,167.024777,0.0,0.00,0.00,0.00,0.0,1.00,11.00,53.79,10000.0
cheating_by_personation_by_using_computer_resource,5322.0,8.755355,138.152790,0.0,0.00,0.00,0.00,0.0,1.00,22.00,103.37,6319.0


... (19 outputs left out)

,column,outliers,outlier_%,Q1,Q3,IQR
39,cheating_420_ipc,197,20.44,0.00,1.00,1.00
9,cheating_personation_using_computer_resource,185,19.19,0.00,2.00,2.00
47,other_offences_bns_ipc_it,181,18.78,0.00,1.00,1.00
12,pub_trans_obscene_material_electronic,146,15.15,0.00,1.00,1.00
8,identity_theft,143,14.83,0.00,1.00,1.00
13,publish_transmit_sexually_explicit_mtrl,95,9.85,0.00,2.00,2.00
3,district_code,0,0.00,182.75,565.00,382.25
4,tampering_computer_source_documents,0,0.00,0.00,0.00,0.00
1,year,0,0.00,2024.00,2024.00,0.00
7,dishonestly_recv_stolen_computer_resource,0,0.00,0.00,0.00,0.00



[8] NUMERIC CORRELATION SCREEN


,feature_1,feature_2,correlation,abs_correlation
1089,gaming_app_website_fraud,marketing_investment_fraud,0.991820,0.991820
1090,gaming_app_website_fraud,fraud_call_vishing,0.988000,0.988000
1105,marketing_investment_fraud,fraud_call_vishing,0.981906,0.981906
1074,otp_frauds,fraud_call_vishing,0.975954,0.975954
1073,otp_frauds,marketing_investment_fraud,0.972014,0.972014
1072,otp_frauds,gaming_app_website_fraud,0.971912,0.971912
1017,credit_card_debit,otp_frauds,0.963031,0.963031
1020,credit_card_debit,fraud_call_vishing,0.924144,0.924144
1022,credit_card_debit,other_frauds,0.921214,0.921214
376,cheating_personation_using_computer_resource,attempt_commit_offences,0.916706,0.916706



[9] TEMPORAL COLUMN SCREEN
Potential temporal columns: ['year']
  year: 1970-01-01 00:00:00.000002024 -> 1970-01-01 00:00:00.000002024 | parse success=100.0%

[10] ML-READINESS OBSERVATIONS — NOT ACTIONS
 - id: very high cardinality / possible identifier
 - year: constant column
 - counterfeiting_stamps: constant column
 - offences_sll_it_lotteries: constant column


## 4. Compact cross-file summary

In [6]:
summary_rows = []

for name, df in datasets.items():
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "numeric_columns": len(
            df.select_dtypes(include=np.number).columns
        ),
        "categorical_columns": len(
            df.select_dtypes(
                include=["object", "string", "category", "bool"]
            ).columns
        ),
        "missing_cells_%": round(
            df.isna().sum().sum() / df.size * 100, 2
        ) if df.size else 0,
        "duplicate_rows": int(df.duplicated().sum()),
    })

display(pd.DataFrame(summary_rows))

,dataset,rows,columns,numeric_columns,categorical_columns,missing_cells_%,duplicate_rows
0,districtwise-cyber-crimes-2017-onwards,5322,48,45,3,0.0,0
1,districtwise-cyber-crimes-2024-onwards,964,56,53,3,0.0,0


# 5. TWO-FILE COMMONALITY / MERGE ANALYSIS

Use this section when you think two datasets may need to be merged.

It checks:

1. Exact common column names
2. Common columns after name normalization
3. Value overlap in common columns
4. Candidate join keys
5. Potential merge risks

It **does not merge the files**.

In [7]:
def compare_two_files(df1, name1, df2, name2):
    print("=" * 110)
    print(f"COMPARISON: {name1}  <->  {name2}")
    print("=" * 110)

    columns1 = list(df1.columns)
    columns2 = list(df2.columns)

    # ------------------------------------------------------------
    # 1. Exact common columns
    # ------------------------------------------------------------
    exact_common = sorted(set(columns1) & set(columns2))

    print("\n[1] EXACT COMMON COLUMNS")
    print(exact_common if exact_common else "None")

    # ------------------------------------------------------------
    # 2. Normalized common columns
    # ------------------------------------------------------------
    normalized1 = {normalize_name(c): c for c in columns1}
    normalized2 = {normalize_name(c): c for c in columns2}

    common_normalized = sorted(
        set(normalized1) & set(normalized2)
    )

    print("\n[2] COMMON COLUMNS AFTER NAME NORMALIZATION")

    if common_normalized:
        normalized_rows = [
            {
                "normalized_name": key,
                "file_1_column": normalized1[key],
                "file_2_column": normalized2[key],
            }
            for key in common_normalized
        ]

        display(pd.DataFrame(normalized_rows))
    else:
        print("None")

    # ------------------------------------------------------------
    # 3. Value overlap
    # ------------------------------------------------------------
    print("\n[3] COMMON-COLUMN VALUE OVERLAP")

    overlap_rows = []

    for key in common_normalized:
        column1 = normalized1[key]
        column2 = normalized2[key]

        series1 = df1[column1].dropna()
        series2 = df2[column2].dropna()

        if len(series1) == 0 or len(series2) == 0:
            continue

        values1 = set(series1.astype(str).str.strip())
        values2 = set(series2.astype(str).str.strip())

        overlap = values1 & values2

        overlap_rows.append({
            "file_1_column": column1,
            "file_2_column": column2,
            "file_1_unique": len(values1),
            "file_2_unique": len(values2),
            "common_unique_values": len(overlap),
            "overlap_%_of_file1": round(
                len(overlap) / len(values1) * 100, 2
            ),
            "overlap_%_of_file2": round(
                len(overlap) / len(values2) * 100, 2
            ),
            "potential_join_key": (
                len(overlap) / len(values1) >= 0.20
                and len(overlap) / len(values2) >= 0.20
            ),
        })

    if overlap_rows:
        display(
            pd.DataFrame(overlap_rows).sort_values(
                "common_unique_values",
                ascending=False
            )
        )
    else:
        print("No comparable common columns with non-null values.")

    # ------------------------------------------------------------
    # 4. Key diagnostics
    # ------------------------------------------------------------
    print("\n[4] POTENTIAL JOIN-KEY DIAGNOSTICS")

    key_rows = []

    for key in common_normalized:
        column1 = normalized1[key]
        column2 = normalized2[key]

        series1 = df1[column1].dropna()
        series2 = df2[column2].dropna()

        if not len(series1) or not len(series2):
            continue

        unique_ratio1 = series1.nunique() / len(series1)
        unique_ratio2 = series2.nunique() / len(series2)

        values1 = set(series1.astype(str).str.strip())
        values2 = set(series2.astype(str).str.strip())

        common_values = len(values1 & values2)

        if common_values:
            key_rows.append({
                "file_1_column": column1,
                "file_2_column": column2,
                "file1_unique_ratio": round(unique_ratio1, 4),
                "file2_unique_ratio": round(unique_ratio2, 4),
                "common_values": common_values,
                "interpretation": (
                    "Inspect semantics, uniqueness and granularity "
                    "before using as a join key."
                ),
            })

    if key_rows:
        display(
            pd.DataFrame(key_rows).sort_values(
                "common_values",
                ascending=False
            )
        )
    else:
        print("No obvious candidate keys found.")

    # ------------------------------------------------------------
    # 5. Merge checklist
    # ------------------------------------------------------------
    print("\n[5] MERGE DECISION CHECKLIST")

    checklist = [
        "Do the common columns represent the same real-world entity?",
        "Are state/district codes using the same coding system?",
        "Are year definitions and coverage periods compatible?",
        "Is the intended relationship one-to-one, one-to-many or many-to-many?",
        "Will the merge multiply rows?",
        "Are similarly named columns actually different concepts?",
        "Are missing keys concentrated in particular years/states/districts?",
        "Does the resulting grain match the intended ML observation?",
    ]

    for item in checklist:
        print(" -", item)

In [8]:
# Automatically compare the first two loaded datasets.
# Change these names if you want a different pair.

if len(datasets) >= 2:
    dataset_names = list(datasets.keys())

    first_name = dataset_names[0]
    second_name = dataset_names[1]

    compare_two_files(
        datasets[first_name],
        first_name,
        datasets[second_name],
        second_name,
    )
else:
    print("Load at least two CSV files to run the two-file analysis.")

COMPARISON: districtwise-cyber-crimes-2017-onwards  <->  districtwise-cyber-crimes-2024-onwards

[1] EXACT COMMON COLUMNS
['cyber_blackmailing_threatening', 'cyber_terrorism', 'data_theft', 'defamation_morphing', 'district_code', 'district_name', 'fake_profile', 'id', 'identity_theft', 'offences_other_than_ransom_ware', 'online_banking_fraud', 'other_frauds', 'other_sll_crimes', 'otp_frauds', 'ransom_ware', 'registration_circles', 'state_code', 'state_name', 'tampering_computer_source_documents', 'year']

[2] COMMON COLUMNS AFTER NAME NORMALIZATION


,normalized_name,file_1_column,file_2_column
0,cyber_blackmailing_threatening,cyber_blackmailing_threatening,cyber_blackmailing_threatening
1,cyber_terrorism,cyber_terrorism,cyber_terrorism
2,data_theft,data_theft,data_theft
3,defamation_morphing,defamation_morphing,defamation_morphing
4,district_code,district_code,district_code
5,district_name,district_name,district_name
6,fake_profile,fake_profile,fake_profile
7,id,id,id
8,identity_theft,identity_theft,identity_theft
9,offences_other_than_ransom_ware,offences_other_than_ransom_ware,offences_other_than_ransom_ware



[3] COMMON-COLUMN VALUE OVERLAP


,file_1_column,file_2_column,file_1_unique,file_2_unique,common_unique_values,overlap_%_of_file1,overlap_%_of_file2,potential_join_key
7,id,id,5322,964,964,18.11,100.00,False
15,registration_circles,registration_circles,939,941,868,92.44,92.24,True
4,district_code,district_code,753,771,748,99.34,97.02,True
5,district_name,district_name,751,768,721,96.01,93.88,True
10,online_banking_fraud,online_banking_fraud,94,53,42,44.68,79.25,True
16,state_code,state_code,36,36,36,100.00,100.00,True
17,state_name,state_name,36,36,36,100.00,100.00,True
8,identity_theft,identity_theft,96,44,34,35.42,77.27,True
9,offences_other_than_ransom_ware,offences_other_than_ransom_ware,113,36,29,25.66,80.56,True
11,other_frauds,other_frauds,78,51,29,37.18,56.86,True



[4] POTENTIAL JOIN-KEY DIAGNOSTICS


,file_1_column,file_2_column,file1_unique_ratio,file2_unique_ratio,common_values,interpretation
7,id,id,1.0000,1.0000,964,"Inspect semantics, uniqueness and granularity before using as a join key."
15,registration_circles,registration_circles,0.1764,0.9761,868,"Inspect semantics, uniqueness and granularity before using as a join key."
4,district_code,district_code,0.1415,0.7998,748,"Inspect semantics, uniqueness and granularity before using as a join key."
5,district_name,district_name,0.1411,0.7967,721,"Inspect semantics, uniqueness and granularity before using as a join key."
10,online_banking_fraud,online_banking_fraud,0.0177,0.0550,42,"Inspect semantics, uniqueness and granularity before using as a join key."
17,state_name,state_name,0.0068,0.0373,36,"Inspect semantics, uniqueness and granularity before using as a join key."
16,state_code,state_code,0.0068,0.0373,36,"Inspect semantics, uniqueness and granularity before using as a join key."
8,identity_theft,identity_theft,0.0180,0.0456,34,"Inspect semantics, uniqueness and granularity before using as a join key."
9,offences_other_than_ransom_ware,offences_other_than_ransom_ware,0.0212,0.0373,29,"Inspect semantics, uniqueness and granularity before using as a join key."
11,other_frauds,other_frauds,0.0147,0.0529,29,"Inspect semantics, uniqueness and granularity before using as a join key."



[5] MERGE DECISION CHECKLIST
 - Do the common columns represent the same real-world entity?
 - Are state/district codes using the same coding system?
 - Are year definitions and coverage periods compatible?
 - Is the intended relationship one-to-one, one-to-many or many-to-many?
 - Will the merge multiply rows?
 - Are similarly named columns actually different concepts?
 - Are missing keys concentrated in particular years/states/districts?
 - Does the resulting grain match the intended ML observation?


# 6. Inspect one column in depth

Use this when a column is important and you want to see its actual values before deciding how to preprocess it.

In [9]:
def inspect_column(df, column, top_n=30):
    if column not in df.columns:
        print(f"Column not found: {column}")
        return

    series = df[column]

    print(f"Column: {column}")
    print(f"dtype: {series.dtype}")
    print(f"non-null: {series.notna().sum():,}")
    print(f"missing: {series.isna().sum():,} ({series.isna().mean() * 100:.2f}%)")
    print(f"unique: {series.nunique(dropna=True):,}")

    print("\nTop values:")
    display(
        series.value_counts(dropna=False).head(top_n).to_frame("frequency")
    )


# Example:
# inspect_column(datasets["crime_women"], "district_name")

# 7. Next stage: ML pipeline

After this notebook, we should **not immediately preprocess everything**.

The recommended sequence is:

```text
RAW CSV(S)
      ↓
DATA UNDERSTANDING                 ← this notebook
      ↓
MERGE / JOIN DECISION              ← only if justified
      ↓
DATA CLEANING
      ↓
EDA + TARGET ANALYSIS
      ↓
FEATURE ENGINEERING
      ↓
TRAIN / VALIDATION / TEST SPLIT
      ↓
BASELINE MODEL
      ↓
MODEL COMPARISON
      ↓
HYPERPARAMETER TUNING
      ↓
EVALUATION
      ↓
ERROR ANALYSIS + EXPLAINABILITY
      ↓
FINAL REPRODUCIBLE PIPELINE
```

The important output from this notebook is the **evidence needed to decide preprocessing**, not preprocessing itself.